# Notebook 3: DiT — Adaptive Layer Normalization

The Diffusion Transformer (DiT) replaces standard layer normalization with **Adaptive Layer Normalization (AdaLN)**, where normalization parameters are predicted from a conditioning signal (timestep, class label, etc.).

In this notebook you will implement:

1. **AdaLN Modulation Function**
2. **DiTBlock** with gated attention and MLP
3. **FinalLayer** with AdaLN projection
4. **Zero-Initialization** verification
5. **Standard vs DiT Block** comparison

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

In [ ]:
# ====== Provided helper modules (from Notebook 1) ======

class Attention(nn.Module):
    """Multi-Head Self-Attention (provided, not a TODO)."""
    def __init__(self, dim, num_heads=8, qkv_bias=False, attn_drop=0.0, proj_drop=0.0):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        x = (attn @ v).transpose(1, 2).reshape(B, T, C)
        return self.proj_drop(self.proj(x))


class Mlp(nn.Module):
    """MLP block (provided, not a TODO)."""
    def __init__(self, in_features, hidden_features=None, drop=0.0):
        super().__init__()
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = nn.GELU(approximate='tanh')
        self.fc2 = nn.Linear(hidden_features, in_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        return self.drop(self.fc2(self.act(self.fc1(x))))

---
## Exercise 3a: AdaLN Modulate Function

The core operation in AdaLN is the **modulate** function:

$$\text{modulate}(x, \text{shift}, \text{scale}) = x \cdot (1 + \text{scale}) + \text{shift}$$

where $x$ has shape `[B, T, D]` and shift/scale have shape `[B, D]`. The shift and scale are broadcast over the sequence dimension $T$.

In [ ]:
def modulate(
    x: torch.Tensor,      # [B, T, D]
    shift: torch.Tensor,   # [B, D]
    scale: torch.Tensor,   # [B, D]
) -> torch.Tensor:
    """
    AdaLN modulation.

    Returns
    -------
    Tensor of shape [B, T, D]
    """
    # TODO: Implement modulate
    # Remember to unsqueeze shift and scale to [B, 1, D] for broadcasting over T
    # Formula: x * (1 + scale) + shift
    raise NotImplementedError("Implement modulate")

In [ ]:
# === Tests for Exercise 3a ===
B, T, D = 2, 5, 8
x = torch.randn(B, T, D)

# Zero shift and zero scale should be identity
shift_z = torch.zeros(B, D)
scale_z = torch.zeros(B, D)
out = modulate(x, shift_z, scale_z)
assert torch.allclose(out, x, atol=1e-6), "Zero shift/scale should be identity"

# Scale of 1 should double the input (before shift)
scale_one = torch.ones(B, D)
out = modulate(x, shift_z, scale_one)
assert torch.allclose(out, 2 * x, atol=1e-6), "Scale=1 should double x"

# Shift only
shift_val = torch.full((B, D), 3.0)
out = modulate(x, shift_val, scale_z)
assert torch.allclose(out, x + 3.0, atol=1e-6), "Shift=3 should add 3"

# Shape check
assert out.shape == (B, T, D)

# Broadcasting: every token in the sequence gets the same modulation
shift = torch.randn(B, D)
scale = torch.randn(B, D)
out = modulate(x, shift, scale)
for t_idx in range(T):
    expected = x[:, t_idx, :] * (1 + scale) + shift
    assert torch.allclose(out[:, t_idx, :], expected, atol=1e-6)

print("Exercise 3a passed!")

---
## Exercise 3b: DiT Block

A DiT block replaces standard LayerNorm with **AdaLN** and adds **gating**. The conditioning vector $c$ (shape `[B, D]`) is mapped to 6 modulation parameters via a linear layer:

$$[\gamma_1, \beta_1, \alpha_1, \gamma_2, \beta_2, \alpha_2] = \text{SiLU}(c) \cdot W + b$$

Then the block computes:

$$x = x + \alpha_1 \cdot \text{Attn}(\text{modulate}(\text{LN}_1(x), \beta_1, \gamma_1))$$
$$x = x + \alpha_2 \cdot \text{MLP}(\text{modulate}(\text{LN}_2(x), \beta_2, \gamma_2))$$

where $\alpha$ are gates, $\beta$ are shifts, $\gamma$ are scales.

The modulation weights are **zero-initialized** so that at initialization the block acts like an identity.

In [ ]:
class DiTBlock(nn.Module):
    """
    DiT Block with Adaptive Layer Normalization.

    Parameters
    ----------
    hidden_size : int
    num_heads : int
    mlp_ratio : float
    """

    def __init__(self, hidden_size: int, num_heads: int, mlp_ratio: float = 4.0):
        super().__init__()
        self.hidden_size = hidden_size

        # TODO: Define the following layers:
        #
        # self.norm1 -> LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)
        # self.norm2 -> LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)
        #
        # self.attn -> Attention(hidden_size, num_heads, qkv_bias=True, attn_drop=0.1)
        # self.mlp  -> Mlp(hidden_size, hidden_features=int(hidden_size * mlp_ratio), drop=0.1)
        #
        # self.adaLN_modulation -> nn.Sequential(
        #     nn.SiLU(),
        #     nn.Linear(hidden_size, 6 * hidden_size, bias=True)
        # )
        #
        # IMPORTANT: Zero-initialize the Linear layer in adaLN_modulation:
        #   nn.init.constant_(self.adaLN_modulation[-1].weight, 0)
        #   nn.init.constant_(self.adaLN_modulation[-1].bias, 0)
        raise NotImplementedError("Define layers")

    def forward(self, x: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        x : Tensor [B, T, hidden_size]  — sequence of tokens
        c : Tensor [B, hidden_size]     — conditioning vector

        Returns
        -------
        Tensor [B, T, hidden_size]
        """
        # TODO:
        # 1. Compute 6 modulation params:
        #    modulation_params = self.adaLN_modulation(c)   -> [B, 6*H]
        #    shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = chunk into 6
        #
        # 2. Attention branch (with gating):
        #    x_norm = modulate(self.norm1(x), shift_msa, scale_msa)
        #    x = x + gate_msa.unsqueeze(1) * self.attn(x_norm)
        #
        # 3. MLP branch (with gating):
        #    x_norm = modulate(self.norm2(x), shift_mlp, scale_mlp)
        #    x = x + gate_mlp.unsqueeze(1) * self.mlp(x_norm)
        #
        # 4. Return x
        raise NotImplementedError("Implement forward")

In [ ]:
# === Tests for Exercise 3b ===
torch.manual_seed(7)
B, T, D, H = 2, 6, 64, 8
dit = DiTBlock(hidden_size=D, num_heads=H).eval()
x = torch.randn(B, T, D)
c = torch.randn(B, D)

out = dit(x, c)

# Shape check
assert out.shape == (B, T, D), f"Expected {(B, T, D)}, got {out.shape}"

# Zero-init check: with zero-initialized modulation, the gates are all zero,
# so the output should be very close to the input (identity at init)
dit_fresh = DiTBlock(hidden_size=D, num_heads=H).eval()
c_zero = torch.zeros(B, D)
out_init = dit_fresh(x, c_zero)
diff = (out_init - x).abs().max().item()
assert diff < 1e-5, f"With zero-init and c=0, block should be ~identity. Max diff: {diff:.2e}"

# Gradient flow through conditioning
dit.train()
c_grad = torch.randn(B, D, requires_grad=True)
x_in = torch.randn(B, T, D)
loss = dit(x_in, c_grad).sum()
loss.backward()
assert c_grad.grad is not None, "Gradients must flow through conditioning"
assert (c_grad.grad.abs() > 0).any(), "Non-zero gradients through c"

# Verify modulation layer dimensions
lin = dit.adaLN_modulation[-1]
assert lin.weight.shape == (6 * D, D), f"Modulation weight shape: {lin.weight.shape}"

print("Exercise 3b passed!")

---
## Exercise 3c: Final Layer with AdaLN

The DiT final layer applies AdaLN modulation before a linear projection to the output dimension:

$$[\text{shift}, \text{scale}] = \text{SiLU}(c) \cdot W + b$$
$$\text{output} = W_{\text{out}} \cdot \text{modulate}(\text{LN}(x), \text{shift}, \text{scale}) + b_{\text{out}}$$

Both the modulation layer and the output linear layer are **zero-initialized**, ensuring near-zero output at initialization.

In [ ]:
class FinalLayer(nn.Module):
    """
    DiT Final Layer: AdaLN + linear projection.

    Parameters
    ----------
    hidden_size : int
    out_dim : int
    """

    def __init__(self, hidden_size: int, out_dim: int):
        super().__init__()
        # TODO: Define:
        # self.norm -> LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)
        # self.adaLN_modulation -> nn.Sequential(SiLU(), Linear(hidden_size, 2*hidden_size))
        # self.linear -> nn.Linear(hidden_size, out_dim)
        #
        # Zero-initialize:
        #   - adaLN_modulation's Linear weight and bias to 0
        #   - self.linear's weight and bias to 0
        raise NotImplementedError("Define layers")

    def forward(self, x: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        x : Tensor [B, T, hidden_size]
        c : Tensor [B, hidden_size]

        Returns
        -------
        Tensor [B, T, out_dim]
        """
        # TODO:
        # 1. shift, scale = self.adaLN_modulation(c).chunk(2, dim=-1)
        # 2. x = modulate(self.norm(x), shift, scale)
        # 3. return self.linear(x)
        raise NotImplementedError("Implement forward")

In [ ]:
# === Tests for Exercise 3c ===
torch.manual_seed(8)
B, T, D, out_dim = 2, 5, 64, 7
final = FinalLayer(hidden_size=D, out_dim=out_dim).eval()
x = torch.randn(B, T, D)
c = torch.randn(B, D)

out = final(x, c)

# Shape check
assert out.shape == (B, T, out_dim), f"Expected {(B, T, out_dim)}, got {out.shape}"

# Zero-init: output should be near zero at initialization
final_fresh = FinalLayer(hidden_size=D, out_dim=out_dim).eval()
out_fresh = final_fresh(x, torch.zeros(B, D))
assert out_fresh.abs().max().item() < 1e-5, (
    f"With zero-init, output should be ~0. Max: {out_fresh.abs().max().item():.2e}"
)

# Modulation layer shape
assert final.adaLN_modulation[-1].weight.shape == (2 * D, D)
assert final.linear.weight.shape == (out_dim, D)

print("Exercise 3c passed!")

---
## Exercise 3d: Verify Zero-Init Properties

Zero-initialization is critical for stable training of DiT models. When the AdaLN modulation weights and biases are initialized to zero:

- All shift, scale, and gate parameters start at 0
- `modulate(x, 0, 0) = x * (1 + 0) + 0 = x` (identity)
- Gate values of 0 zero out the attention/MLP contributions
- The model starts as an identity function and gradually learns to modify its output

Implement a function that verifies these properties.

In [ ]:
def verify_zero_init(block: DiTBlock) -> dict:
    """
    Verify that a freshly initialized DiTBlock has zero-initialized modulation.

    Returns
    -------
    dict with keys:
        'mod_weight_is_zero' : bool
        'mod_bias_is_zero' : bool
        'all_modulation_params_zero' : bool  (when c=0, all 6 params should be 0)
        'identity_at_init' : bool  (output ≈ input when c=0)
    """
    # TODO:
    # 1. Check that the Linear layer in adaLN_modulation has all-zero weight
    # 2. Check that the Linear layer in adaLN_modulation has all-zero bias
    # 3. Pass c=0 through adaLN_modulation and verify output is all zeros
    # 4. Pass (x, c=0) through the block and verify output ≈ input
    raise NotImplementedError("Implement verify_zero_init")

In [ ]:
# === Tests for Exercise 3d ===
torch.manual_seed(9)
D, H = 64, 8
fresh_block = DiTBlock(hidden_size=D, num_heads=H).eval()

result = verify_zero_init(fresh_block)

assert result['mod_weight_is_zero'], "Modulation weight should be zero at init"
assert result['mod_bias_is_zero'], "Modulation bias should be zero at init"
assert result['all_modulation_params_zero'], "All 6 modulation params should be 0 when c=0"
assert result['identity_at_init'], "Block should act as identity at init (c=0)"

print("Exercise 3d passed!")

---
## Exercise 3e: Standard Block vs DiT Block Comparison

Compare the behavior of a standard Pre-LN Transformer block with a DiT block.

Implement:
1. A standard `TransformerBlock` (pre-LN, no conditioning)
2. Run both blocks on the same input
3. Verify that DiT with zero conditioning produces output close to input (identity), while the standard block does not

In [ ]:
class StandardTransformerBlock(nn.Module):
    """
    Standard pre-LN Transformer block (no conditioning).

    Parameters
    ----------
    hidden_size : int
    num_heads : int
    mlp_ratio : float
    """

    def __init__(self, hidden_size: int, num_heads: int, mlp_ratio: float = 4.0):
        super().__init__()
        # TODO: Define:
        # self.norm1 -> LayerNorm(hidden_size)
        # self.norm2 -> LayerNorm(hidden_size)
        # self.attn  -> Attention(hidden_size, num_heads, qkv_bias=True, attn_drop=0.1)
        # self.mlp   -> Mlp(hidden_size, hidden_features=int(hidden_size * mlp_ratio), drop=0.1)
        raise NotImplementedError("Define layers")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Standard pre-LN forward (no conditioning)
        # x = x + self.attn(self.norm1(x))
        # x = x + self.mlp(self.norm2(x))
        raise NotImplementedError("Implement forward")

In [ ]:
def compare_blocks(
    standard_block: StandardTransformerBlock,
    dit_block: DiTBlock,
    x: torch.Tensor,
) -> dict:
    """
    Compare outputs of a standard block and a DiT block.

    Returns
    -------
    dict with keys:
        'standard_modifies_input' : bool  (|output - input| > threshold)
        'dit_zero_is_identity' : bool  (DiT with c=0 gives output ≈ input)
        'dit_nonzero_modifies_input' : bool  (DiT with c!=0 modifies input)
    """
    # TODO:
    # 1. Run standard_block(x), check if output differs from input
    # 2. Run dit_block(x, c=zeros), check if output ≈ input
    # 3. Run dit_block(x, c=randn), check if output differs from input
    # Threshold for "modifies": max absolute difference > 1e-4
    raise NotImplementedError("Implement compare_blocks")

In [ ]:
# === Tests for Exercise 3e ===
torch.manual_seed(11)
D, H = 64, 8
B, T = 2, 6

std_block = StandardTransformerBlock(D, H).eval()
dit_block = DiTBlock(D, H).eval()
x = torch.randn(B, T, D)

result = compare_blocks(std_block, dit_block, x)

assert result['standard_modifies_input'], (
    "Standard block should modify input (it has random weights, not identity)"
)
assert result['dit_zero_is_identity'], (
    "DiT block with c=0 should be identity at init (zero-init gates)"
)
assert result['dit_nonzero_modifies_input'], (
    "DiT block with non-zero c should modify input (conditioning activates gates)"
)

# Standard block output shape
std_out = std_block(x)
assert std_out.shape == (B, T, D)

print("Exercise 3e passed!")
print("\n=== All Notebook 3 exercises passed! ===")